# Freight Rate Prediction & Explainable AI (XAI)

This notebook performs Exploratory Data Analysis (EDA), builds an XGBoost model to predict freight load rates, and implements **Explainable AI (XAI)** techniques (Global Feature Gain, SHAP Summary, and SHAP Local Waterfall breakdowns) to explain model decisions.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import shap
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.impute import SimpleImputer

os.makedirs('output', exist_ok=True)

sns.set_theme(style='whitegrid')

## 1. Data Loading

In [ ]:
train_df = pd.read_csv('train-test.csv')
val_df = pd.read_csv('validation.csv')
dec_df = pd.read_csv('december-chart-inputs.csv')

train_df['date'] = pd.to_datetime(train_df['date'])
val_df['date'] = pd.to_datetime(val_df['date'])

display(train_df.head())

## 2. Exploratory Data Analysis (EDA)

### 2.1 Target Variable Distribution
Let's look at the distribution of our target variable `posted_rate`.

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(train_df['posted_rate'], bins=50, kde=True)
plt.title('Distribution of Posted Rates')
plt.xlabel('Posted Rate ($)')
plt.ylabel('Frequency')
plt.tight_layout()
plt.savefig('output/eda_distribution.png')
plt.show()

### 2.2 Rate by Equipment Type
Different truck equipment types usually command different rates.

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(x='equipment', y='posted_rate', data=train_df)
plt.title('Posted Rate by Equipment Type')
plt.tight_layout()
plt.savefig('output/eda_equipment.png')
plt.show()

### 2.3 Distance vs Rate
We expect a strong positive correlation between distance and the total posted rate.

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(x='distance', y='posted_rate', hue='equipment', alpha=0.3, data=train_df)
plt.title('Distance vs Posted Rate')
plt.tight_layout()
plt.savefig('output/eda_distance.png')
plt.show()

## 3. Feature Engineering
We extract time-based components and handle missing values required for the model.

In [ ]:
def engineer_features(df, is_december=False):
    df = df.copy()
    df['date'] = pd.to_datetime(df['date'])
    df['month'] = df['date'].dt.month
    df['day_of_week'] = df['date'].dt.dayofweek
    df['day_of_month'] = df['date'].dt.day
    
    if is_december:
        for col in ['market_index', 'quote_signal', 'pickup_lat', 'pickup_lon', 'delivery_lat', 'delivery_lon']:
            df[col] = np.nan
            
    cat_cols = ['pickup', 'delivery', 'equipment']
    for col in cat_cols:
        df[col] = df[col].astype('category')
        
    return df

train_feat = engineer_features(train_df)
val_feat = engineer_features(val_df)
dec_feat = engineer_features(dec_df, is_december=True)

## 4. Imputation and Time-based Split

In [ ]:
features = [
    'pickup', 'delivery', 'distance', 'equipment', 'weight',
    'market_index', 'quote_signal', 
    'month', 'day_of_week', 'day_of_month',
    'pickup_lat', 'pickup_lon', 'delivery_lat', 'delivery_lon'
]
target = 'posted_rate'

num_cols = ['weight', 'market_index', 'quote_signal', 'pickup_lat', 'pickup_lon', 'delivery_lat', 'delivery_lon']
imputer = SimpleImputer(strategy='median')
imputer.fit(train_feat[num_cols])

train_feat[num_cols] = imputer.transform(train_feat[num_cols])
val_feat[num_cols] = imputer.transform(val_feat[num_cols])
dec_feat[num_cols] = imputer.transform(dec_feat[num_cols])

train_split = train_feat[train_feat['date'] < '2025-10-01']
val_split = train_feat[train_feat['date'] >= '2025-10-01']

X_train_split, y_train_split = train_split[features], train_split[target]
X_val_split, y_val_split = val_split[features], val_split[target]

## 5. Model Training and Validation
We train an XGBoost model and validate on the Oct hold-out set.

In [ ]:
model = xgb.XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    enable_categorical=True,
    random_state=42
)

model.fit(X_train_split, y_train_split, eval_set=[(X_val_split, y_val_split)], verbose=50)

val_preds = model.predict(X_val_split)
print('Validation RMSE:', np.sqrt(mean_squared_error(y_val_split, val_preds)))
print('Validation MAE:', mean_absolute_error(y_val_split, val_preds))

## 6. Explainable AI (XAI)
Here we use tree feature importance and SHAP (SHapley Additive exPlanations) to explain global feature contributions and individual load predictions.

In [ ]:
# 6.1 Global Feature Importance (Gain Metric)
importance = model.get_booster().get_score(importance_type='gain')
importance_df = pd.DataFrame({
    'Feature': list(importance.keys()),
    'Gain': list(importance.values())
}).sort_values(by='Gain', ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(importance_df['Feature'], importance_df['Gain'], color='#2b5c8f')
plt.title('XAI: Global Feature Importance (Average Gain)')
plt.xlabel('Gain (Improvement in Loss)')
plt.tight_layout()
plt.savefig('output/xai_feature_importance.png', dpi=300)
plt.show()

In [ ]:
# 6.2 SHAP Global Feature Impact (Summary / Beeswarm Plot)
# Using native XGBoost TreeSHAP (faster & version robust)
sample_val = X_val_split.sample(n=min(1000, len(X_val_split)), random_state=42)
dmat_sample = xgb.DMatrix(sample_val, enable_categorical=True)
contribs = model.get_booster().predict(dmat_sample, pred_contribs=True)

shap_values = contribs[:, :-1]
base_values = contribs[:, -1]

shap_explanation = shap.Explanation(
    values=shap_values,
    base_values=base_values,
    data=sample_val,
    feature_names=features
)

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, sample_val, show=False)
plt.title('XAI: SHAP Summary (Directional Feature Impact)', fontsize=14, pad=12)
plt.tight_layout()
plt.savefig('output/xai_shap_summary.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 6.3 SHAP Local Explanation (Waterfall Plot for a Single Load)
# Explain how the model arrived at the prediction for the first sample load
plt.figure(figsize=(10, 6))
shap.plots.waterfall(shap_explanation[0], show=False)
plt.title('XAI: SHAP Local Prediction Breakdown (Sample Load)', fontsize=14, pad=12)
plt.tight_layout()
plt.savefig('output/xai_shap_waterfall.png', dpi=300, bbox_inches='tight')
plt.show()

## 7. Final Predictions
Retrain on the entire training data and predict on the hold-out validation and December data.

In [ ]:
print('Retraining on full training data...')
model_full = xgb.XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    enable_categorical=True,
    random_state=42
)
model_full.fit(train_feat[features], train_feat[target])

print('Predicting on validation.csv')
final_val_preds = model_full.predict(val_feat[features])
val_out = val_df[['load_id']].copy()
val_out['predicted_rate'] = final_val_preds
val_out.to_csv('output/validation_predictions.csv', index=False)

print('Predicting on december-chart-inputs.csv')
dec_preds = model_full.predict(dec_feat[features])
dec_out = dec_df.copy()
dec_out['predicted_rate'] = dec_preds
dec_out.to_csv('output/december-chart-inputs.csv', index=False)
print('Predictions saved successfully in output/ directory!')